In [355]:
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## Wingbacks

In [356]:
# Creating the synthetic data

# Create an empty DataFrame with the specified columns
columns=["goals", "assists", "shots", "shot_accuracy", "passes", "pass_accuracy", "dribbles", "dribble_success_rate", "tackles", "tackle_success_rate", "offsides", "fouls_committed", "possession_won", "possession_lost", "minutes_played", "distance_covered", "distance_sprinted", "half_length", "team_xg", "opponent_xg", "opponent_goals"]
pos_df=pd.DataFrame(columns=columns)

# Adding example wingback data manually for 4 performances
wingback_test_data = [
    {
        "minutes_played": 90, "half_length": 10,
        "goals": 1, "assists": 1, "shots": 2, "shot_accuracy": 50.0,
        "passes": 28, "pass_accuracy": 82.0, 
        "dribbles": 26, "dribble_success_rate": 78.0, 
        "tackles": 5, "tackle_success_rate": 60.0, 
        "possession_won": 5, "possession_lost": 4, 
        "distance_covered": 14.8, "distance_sprinted": 8.5,
        "fouls_committed": 1, "offsides": 0,
        "opponent_goals": 0, "team_xg": 3.2, "opponent_xg": 0.5 # Triggers Clean Sheet Lockdown
    },
    {
        "minutes_played": 55, "half_length": 10,
        "goals": 0, "assists": 0, "shots": 0, "shot_accuracy": 0.0,
        "passes": 8, "pass_accuracy": 62.0, 
        "dribbles": 6, "dribble_success_rate": 25.0, 
        "tackles": 2, "tackle_success_rate": 15.0, 
        "possession_won": 1, "possession_lost": 6, 
        "distance_covered": 5.8, "distance_sprinted": 1.5,
        "fouls_committed": 2, "offsides": 0,
        "opponent_goals": 3, "team_xg": 1.1, "opponent_xg": 2.8 # Triggers Defensive Meltdown
    },
    {
        "minutes_played": 90, "half_length": 10,
        "goals": 0, "assists": 0, "shots": 0, "shot_accuracy": 0.0,
        "passes": 14, "pass_accuracy": 90.0, 
        "dribbles": 5, "dribble_success_rate": 100.0, 
        "tackles": 1, "tackle_success_rate": 50.0, 
        "possession_won": 1, "possession_lost": 1, 
        "distance_covered": 10.2, "distance_sprinted": 2.1,
        "fouls_committed": 0, "offsides": 0,
        "opponent_goals": 1, "team_xg": 1.5, "opponent_xg": 1.0
    },
    {
        "minutes_played": 90, "half_length": 10,
        "goals": 0, "assists": 0, "shots": 0, "shot_accuracy": 0.0,
        "passes": 22, "pass_accuracy": 88.0, 
        "dribbles": 3, "dribble_success_rate": 66.0, 
        "tackles": 8, "tackle_success_rate": 75.0, 
        "possession_won": 7, "possession_lost": 2, 
        "distance_covered": 11.0, "distance_sprinted": 3.0,
        "fouls_committed": 0, "offsides": 0,
        "opponent_goals": 1, "team_xg": 1.2, "opponent_xg": 1.2
    }
]
# Add the test data to the empty dataframe (column names match the keys in the dictionaries)
for i in range(len(wingback_test_data)):
    pos_df.loc[i] = wingback_test_data[i]

print(pos_df)

   goals  assists  shots  shot_accuracy  passes  pass_accuracy  dribbles  \
0      1        1      2           50.0      28           82.0        26   
1      0        0      0            0.0       8           62.0         6   
2      0        0      0            0.0      14           90.0         5   
3      0        0      0            0.0      22           88.0         3   

   dribble_success_rate  tackles  tackle_success_rate  ...  fouls_committed  \
0                  78.0        5                 60.0  ...                1   
1                  25.0        2                 15.0  ...                2   
2                 100.0        1                 50.0  ...                0   
3                  66.0        8                 75.0  ...                0   

   possession_won  possession_lost  minutes_played  distance_covered  \
0               5                4              90              14.8   
1               1                6              55               5.8   
2      

In [357]:
# 2. Force them all to numeric in one swoop
cols_to_convert = pos_df.columns 
pos_df[cols_to_convert] = pos_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
pos_df[cols_to_convert] = pos_df[cols_to_convert].fillna(0)

In [358]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = pos_df[pos_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    pos_df[perc_col] = pos_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

In [359]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    pos_df[col] = pos_df[col] * (h_base / pos_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    pos_df[f"{col}_p90"] = (pos_df[col] / (pos_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = means_stds_dict[f"{col}_p90"]["mean"]
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    pos_df[f"{col}_p90"] = ((pos_df[col] + dummy_stat) / (pos_df["minutes_played"] + dummy_minutes)) * 90.0

In [360]:
pos_df["xt_bonus_p90"] = 0.25 * (pos_df["distance_sprinted_p90"] / pos_df["distance_covered_p90"]) * np.log(pos_df["pass_accuracy"] * pos_df["passes_p90"] + 1)

In [361]:
# Match supremacy scalar
gamma = 0.2
delta_xg = (pos_df['team_xg'] + 1) / (pos_df['opponent_xg'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

In [362]:
# Now we import weights from root/workshop/position_weights.json
# Which is in the format {"RWB": {"goals_p90": 0.3, ...}, "CM": {...}, "ST": {...}}
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xt_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["RWB"]
final_weights = np.array([weights_dict[col] for col in col_names])

In [363]:
# Now we import the means and stds from root/workshop/position_means_stds.json
# Which is in the format {"CB": {"goals_p90": {"mean": 0.1, "std": 0.2}, ...}, "CM": {...}, "ST": {...}}
position_means_stds_path = project_root / "workshop" / "position_means_stds.json"
with open(position_means_stds_path, "r") as f:
    position_means_stds = json.load(f)
means_stds_dict = position_means_stds["RWB"]

In [364]:
# Compute final z-scores
negative_stats = ["fouls_committed_p90", "possession_lost_p90", "offsides_p90"]
for col in col_names:
    mean = means_stds_dict[col]["mean"]
    std = means_stds_dict[col]["std"]
    if std == 0:
        pos_df[f"{col}_z"] = 0
    elif col in negative_stats:
        pos_df[f"{col}_z"] = (mean - pos_df[col]) / std
    else:
        pos_df[f"{col}_z"] = (pos_df[col] - mean) / std

In [365]:
# ---------------------------------------------------------
# Step: Volume Masking (The Z-Score Override)
# If a player doesn't attempt enough actions, we neutralize 
# their efficiency Z-score to 0.0 so it doesn't affect their rating.
# ---------------------------------------------------------
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

# Map the raw volume column to its corresponding Z-score column
volume_z_pairs = {
    "passes": "pass_accuracy_z",
    "shots": "shot_accuracy_z",
    "dribbles": "dribble_success_rate_z",
    "tackles": "tackle_success_rate_z"
}

for vol_col, z_col in volume_z_pairs.items():
    # np.where(condition, value_if_true, value_if_false)
    # If volume < threshold, force Z-score to 0.0. Otherwise, keep the original Z-score.
    pos_df[z_col] = np.where(pos_df[vol_col] < volume_masks[vol_col], 0.0, pos_df[z_col])

In [366]:
# Attacking Floor: No penalty for 0 shots/goals
attacking_stats = ['goals_p90_z', 'assists_p90_z', 'shots_p90_z', 'shot_accuracy_z']
for col in attacking_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=0.0, a_max=None)

# Efficiency Volatility Floor
efficiency_stats = ['dribble_success_rate_z', 'pass_accuracy_z', 'tackle_success_rate_z']
for col in efficiency_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.75, a_max=None)

# The Active Detriment Cap
# Punishes them for bad turnovers/fouls, but stops the free-fall at -1.5
detriment_stats = ['fouls_committed_p90_z', 'possession_lost_p90_z']

for col in detriment_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-1.50, a_max=None)

# ---------------------------------------------------------
# The Passenger Cap (Volume Floors)
# Volume stats measure involvement. If they are a passenger, cap the penalty at -1.0 
# so they aren't mathematically treated as an active threat to the team.
# ---------------------------------------------------------
passenger_stats = [
    'passes_p90_z', 'dribbles_p90_z', 'tackles_p90_z', 
    'possession_won_p90_z', 'distance_covered_p90_z', 
    'distance_sprinted_p90_z', 'xt_bonus_p90_z'
]

for col in passenger_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.5, a_max=None)

# NOTE: Active Detriment stats (possession lost, fouls, offsides) 
# DO NOT get this cap, because they actively hurt the team.

In [367]:
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = pos_df[z_score_cols].values
pos_df['raw_score'] = np.dot(z_matrix, final_weights)

In [368]:
# Goal Contributions (Standard)
pos_df['raw_score'] += pos_df['goals'] * 0.6
pos_df['raw_score'] += pos_df['assists'] * 0.8

pos_df['raw_score'] += np.where(pos_df['distance_covered_p90_z'] > 1.5, (pos_df['distance_covered_p90_z'] - 1.5) * 0.15, 0)
pos_df['raw_score'] += np.where(pos_df['distance_sprinted_p90_z'] > 1.5, (pos_df['distance_sprinted_p90_z'] - 1.5) * 0.15, 0)
pos_df['raw_score'] += np.where(pos_df['xt_bonus_p90_z'] > 1.5, (pos_df['xt_bonus_p90_z'] - 1.5) * 0.15, 0)

# ---------------------------------------------------------
# The "Third CB" Defensive Shift Bonus
# If a Wingback puts in an elite defensive shift (sacrificing their attack), 
# we give them a raw score bump so they aren't punished for playing defensively.
# ---------------------------------------------------------
defensive_masterclass = (pos_df['tackles_p90_z'] > 1.0) & (pos_df['possession_won_p90_z'] > 1.0)

# Add +0.35 to the raw score (This translates to roughly +0.4 to +0.5 on the final rating)
pos_df['raw_score'] += np.where(defensive_masterclass, 0.35, 0)

In [369]:
# 1. Base Eligibility
played_enough = pos_df['minutes_played'] >= 60
clean_sheet = pos_df['opponent_goals'] == 0

# 2. The "Lockdown" Masterclass
# They kept a clean sheet AND suffocated the opponent (e.g., at or under 1.0 xG)
lockdown_mask = played_enough & clean_sheet & (pos_df['opponent_xg'] <= 1.0)

# 3. The "Goalkeeper Bailout"
# They kept a clean sheet, but the backline was heavily breached (e.g., over 2.0 xG)
bailout_mask = played_enough & clean_sheet & (pos_df['opponent_xg'] >= 2.0)

# 4. Standard Clean Sheet
# Everything in between 1.0 and 2.0 xG
standard_cs_mask = played_enough & clean_sheet & ~lockdown_mask & ~bailout_mask

# Apply the tiered rewards to the raw score:
# Lockdown: +0.5 (Massive reward for total prevention)
pos_df['raw_score'] += np.where(lockdown_mask, 0.5, 0)

# Standard: +0.35 (The normal reward)
pos_df['raw_score'] += np.where(standard_cs_mask, 0.35, 0)

# Bailed Out: +0.15 (They get a tiny bump for surviving, but the GK was the hero)
pos_df['raw_score'] += np.where(bailout_mask, 0.15, 0)

In [370]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
pos_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (pos_df['raw_score'] - s_0))))

In [371]:
pos_df['final_rating'] = pos_df['raw_rating'] - match_supremacy_scalar

In [372]:
pos_df['final_rating'] = pos_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [373]:
random_row = pos_df.sample(n=1).iloc[0]
for col in pos_df.columns:
    print(f"{col}: {random_row[col]:.2f},")

goals: 0.00,
assists: 0.00,
shots: 0.00,
shot_accuracy: 50.00,
passes: 14.00,
pass_accuracy: 90.00,
dribbles: 5.00,
dribble_success_rate: 100.00,
tackles: 1.00,
tackle_success_rate: 50.00,
offsides: 0.00,
fouls_committed: 0.00,
possession_won: 1.00,
possession_lost: 1.00,
minutes_played: 90.00,
distance_covered: 10.20,
distance_sprinted: 2.10,
half_length: 10.00,
team_xg: 1.50,
opponent_xg: 1.00,
opponent_goals: 1.00,
goals_p90: 0.00,
assists_p90: 0.00,
shots_p90: 0.00,
passes_p90: 20.54,
dribbles_p90: 12.15,
tackles_p90: 1.01,
possession_won_p90: 0.97,
possession_lost_p90: 2.44,
fouls_committed_p90: 0.05,
offsides_p90: 0.01,
distance_covered_p90: 10.19,
distance_sprinted_p90: 2.53,
xt_bonus_p90: 0.47,
goals_p90_z: 0.00,
assists_p90_z: 0.00,
shots_p90_z: 0.00,
shot_accuracy_z: 0.00,
passes_p90_z: 0.28,
pass_accuracy_z: 0.18,
dribbles_p90_z: -0.50,
dribble_success_rate_z: 2.59,
tackles_p90_z: -0.50,
tackle_success_rate_z: 0.00,
offsides_p90_z: -0.02,
fouls_committed_p90_z: 0.33,
possess

## Wide Midfielders

In [374]:
# Creating the synthetic data

# Create an empty DataFrame with the specified columns
columns=["player", "goals", "assists", "shots", "shot_accuracy", "passes", "pass_accuracy", "dribbles", "dribble_success_rate", "tackles", "tackle_success_rate", "offsides", "fouls_committed", "possession_won", "possession_lost", "minutes_played", "distance_covered", "distance_sprinted", "half_length", "team_xg", "opponent_xg", "opponent_goals"]
pos_df=pd.DataFrame(columns=columns)

# Adding example wingback data manually for 4 performances
wm_test_data = [
    {
        "player": "The Masterclass (The Prime Beckham)",
        "minutes_played": 90, "half_length": 10,
        "goals": 0, "assists": 2, "shots": 1, "shot_accuracy": 100.0,
        "passes": 38, "pass_accuracy": 85.0, 
        "dribbles": 6, "dribble_success_rate": 80.0, 
        "tackles": 6, "tackle_success_rate": 66.0, 
        "possession_won": 5, "possession_lost": 3, 
        "distance_covered": 13.5, "distance_sprinted": 5.5,
        "fouls_committed": 1, "offsides": 0,
        "opponent_goals": 0, "team_xg": 2.5, "opponent_xg": 0.8
    },
    {
        "player": "The Disaster (The Liability)",
        "minutes_played": 55, "half_length": 10,
        "goals": 0, "assists": 0, "shots": 0, "shot_accuracy": 0.0,
        "passes": 12, "pass_accuracy": 65.0, 
        "dribbles": 4, "dribble_success_rate": 25.0, 
        "tackles": 0, "tackle_success_rate": 0.0, 
        "possession_won": 1, "possession_lost": 6, 
        "distance_covered": 6.1, "distance_sprinted": 1.5,
        "fouls_committed": 2, "offsides": 0,
        "opponent_goals": 2, "team_xg": 1.0, "opponent_xg": 2.0
    },
    {
        "player": "The Ghost (The Passenger)",
        "minutes_played": 90, "half_length": 10,
        "goals": 0, "assists": 0, "shots": 0, "shot_accuracy": 0.0,
        "passes": 16, "pass_accuracy": 88.0, 
        "dribbles": 2, "dribble_success_rate": 100.0, 
        "tackles": 1, "tackle_success_rate": 100.0, 
        "possession_won": 1, "possession_lost": 2, 
        "distance_covered": 10.5, "distance_sprinted": 2.2,
        "fouls_committed": 0, "offsides": 0,
        "opponent_goals": 1, "team_xg": 1.2, "opponent_xg": 1.2
    },
    {
        "player": "The Wrong Blueprint (The Winger trapped at RM)",
        "minutes_played": 90, "half_length": 10,
        "goals": 1, "assists": 0, "shots": 4, "shot_accuracy": 50.0,
        "passes": 15, "pass_accuracy": 75.0, 
        "dribbles": 14, "dribble_success_rate": 60.0, 
        "tackles": 0, "tackle_success_rate": 0.0, 
        "possession_won": 1, "possession_lost": 7, 
        "distance_covered": 11.2, "distance_sprinted": 5.0,
        "fouls_committed": 0, "offsides": 2,
        "opponent_goals": 2, "team_xg": 1.8, "opponent_xg": 1.5
    }
]
# Add the test data to the empty dataframe (column names match the keys in the dictionaries)
for i in range(len(wm_test_data)):
    pos_df.loc[i] = wm_test_data[i]

print(pos_df)

                                           player  goals  assists  shots  \
0             The Masterclass (The Prime Beckham)      0        2      1   
1                    The Disaster (The Liability)      0        0      0   
2                       The Ghost (The Passenger)      0        0      0   
3  The Wrong Blueprint (The Winger trapped at RM)      1        0      4   

   shot_accuracy  passes  pass_accuracy  dribbles  dribble_success_rate  \
0          100.0      38           85.0         6                  80.0   
1            0.0      12           65.0         4                  25.0   
2            0.0      16           88.0         2                 100.0   
3           50.0      15           75.0        14                  60.0   

   tackles  ...  fouls_committed  possession_won  possession_lost  \
0        6  ...                1               5                3   
1        0  ...                2               1                6   
2        1  ...                0    

In [375]:
# 2. Force them all to numeric in one swoop
cols_to_convert = pos_df.columns
# exempt the 'player' column since it's text
cols_to_convert = cols_to_convert.drop('player') 
pos_df[cols_to_convert] = pos_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
pos_df[cols_to_convert] = pos_df[cols_to_convert].fillna(0)

In [376]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = pos_df[pos_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    pos_df[perc_col] = pos_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

In [377]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    pos_df[col] = pos_df[col] * (h_base / pos_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    pos_df[f"{col}_p90"] = (pos_df[col] / (pos_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = means_stds_dict[f"{col}_p90"]["mean"]
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    pos_df[f"{col}_p90"] = ((pos_df[col] + dummy_stat) / (pos_df["minutes_played"] + dummy_minutes)) * 90.0

In [378]:
pos_df["xt_bonus_p90"] = 0.25 * (pos_df["distance_sprinted_p90"] / pos_df["distance_covered_p90"]) * np.log(pos_df["pass_accuracy"] * pos_df["passes_p90"] + 1)

In [379]:
# Match supremacy scalar
gamma = 0.2
delta_xg = (pos_df['team_xg'] + 1) / (pos_df['opponent_xg'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

In [380]:
# Now we import weights from root/workshop/position_weights.json
# Which is in the format {"RWB": {"goals_p90": 0.3, ...}, "CM": {...}, "ST": {...}}
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xt_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["RM"]
final_weights = np.array([weights_dict[col] for col in col_names])

In [381]:
# Now we import the means and stds from root/workshop/position_means_stds.json
# Which is in the format {"CB": {"goals_p90": {"mean": 0.1, "std": 0.2}, ...}, "CM": {...}, "ST": {...}}
position_means_stds_path = project_root / "workshop" / "position_means_stds.json"
with open(position_means_stds_path, "r") as f:
    position_means_stds = json.load(f)
means_stds_dict = position_means_stds["RM"]

In [382]:
# Compute final z-scores
negative_stats = ["fouls_committed_p90", "possession_lost_p90", "offsides_p90"]
for col in col_names:
    mean = means_stds_dict[col]["mean"]
    std = means_stds_dict[col]["std"]
    if std == 0:
        pos_df[f"{col}_z"] = 0
    elif col in negative_stats:
        pos_df[f"{col}_z"] = (mean - pos_df[col]) / std
    else:
        pos_df[f"{col}_z"] = (pos_df[col] - mean) / std

In [383]:
# ---------------------------------------------------------
# Step: Volume Masking (The Z-Score Override)
# If a player doesn't attempt enough actions, we neutralize 
# their efficiency Z-score to 0.0 so it doesn't affect their rating.
# ---------------------------------------------------------
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

# Map the raw volume column to its corresponding Z-score column
volume_z_pairs = {
    "passes": "pass_accuracy_z",
    "shots": "shot_accuracy_z",
    "dribbles": "dribble_success_rate_z",
    "tackles": "tackle_success_rate_z"
}

for vol_col, z_col in volume_z_pairs.items():
    # np.where(condition, value_if_true, value_if_false)
    # If volume < threshold, force Z-score to 0.0. Otherwise, keep the original Z-score.
    pos_df[z_col] = np.where(pos_df[vol_col] < volume_masks[vol_col], 0.0, pos_df[z_col])

In [384]:
# ---------------------------------------------------------
# 1. TACTICAL FLOORS (What an RM/LM is exempt from)
# They are wide playmakers, not pure strikers. Do not punish them for not shooting.
# ---------------------------------------------------------
attacking_stats = ['goals_p90_z', 'assists_p90_z', 'shots_p90_z', 'shot_accuracy_z']
for col in attacking_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.5, a_max=None)

# ---------------------------------------------------------
# 2. STANDARD CAPS (The Safety Nets)
# ---------------------------------------------------------
# Efficiency Volatility
efficiency_stats = ['dribble_success_rate_z', 'pass_accuracy_z', 'tackle_success_rate_z']
for col in efficiency_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.75, a_max=None)

# Active Detriments (Fouls, Turnovers, Offsides)
detriment_stats = ['fouls_committed_p90_z', 'possession_lost_p90_z', 'offsides_p90_z']
for col in detriment_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-1.50, a_max=None)

# Passenger Cap (Volume Floors)
passenger_stats = [
    'passes_p90_z', 'dribbles_p90_z', 'tackles_p90_z', 
    'possession_won_p90_z', 'distance_covered_p90_z', 
    'distance_sprinted_p90_z', 'xt_bonus_p90_z'
]
for col in passenger_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.50, a_max=None)

In [385]:
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = pos_df[z_score_cols].values
pos_df['raw_score'] = np.dot(z_matrix, final_weights)

In [386]:
# ---------------------------------------------------------
# 3. CONTEXTUAL BONUSES (The RM/LM Identity)
# ---------------------------------------------------------
# Calculate Raw Score first
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = pos_df[z_score_cols].values
pos_df['raw_score'] = np.dot(z_matrix, final_weights)

# Standard G/A Bonus
pos_df['raw_score'] += pos_df['goals'] * 0.6
pos_df['raw_score'] += pos_df['assists'] * 0.8

# THE "TWO-WAY HUB" BONUS
# An RM/LM is judged on their ability to dictate play AND track back.
two_way_master = (pos_df['passes_p90_z'] > 1.0) & (pos_df['tackles_p90_z'] > 1.0)
pos_df['raw_score'] += np.where(two_way_master, 0.40, 0)

# WIDE PLAYMAKER THREAT
# Reward them for stretching the defense and hitting dangerous crosses
pos_df['raw_score'] += np.where(pos_df['xt_bonus_p90_z'] > 1.5, (pos_df['xt_bonus_p90_z'] - 1.5) * 0.15, 0)

# MIDFIELD CLEAN SHEET BUMP
# Midfielders get a flat +0.15 for helping secure a clean sheet (no complex lockdown logic needed here)
clean_sheet_mask = (pos_df['opponent_goals'] == 0) & (pos_df['minutes_played'] >= 60)
pos_df['raw_score'] += np.where(clean_sheet_mask, 0.15, 0)

In [387]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
pos_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (pos_df['raw_score'] - s_0))))

In [388]:
pos_df['final_rating'] = pos_df['raw_rating'] - match_supremacy_scalar

In [389]:
pos_df['final_rating'] = pos_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [390]:
random_row = pos_df.sample(n=1).iloc[0]
for col in pos_df.columns:
    if isinstance(random_row[col], float):
        print(f"{col}: {random_row[col]:.2f},")
    else:
        print(f"{col}: {random_row[col]},")

player: The Ghost (The Passenger),
goals: 0.00,
assists: 0.00,
shots: 0.00,
shot_accuracy: 50.00,
passes: 16.00,
pass_accuracy: 88.00,
dribbles: 2.00,
dribble_success_rate: 55.00,
tackles: 1.00,
tackle_success_rate: 66.00,
offsides: 0.00,
fouls_committed: 0.00,
possession_won: 1.00,
possession_lost: 2.00,
minutes_played: 90,
distance_covered: 10.50,
distance_sprinted: 2.20,
half_length: 10,
team_xg: 1.20,
opponent_xg: 1.20,
opponent_goals: 1,
goals_p90: 0.00,
assists_p90: 0.00,
shots_p90: 0.00,
passes_p90: 16.78,
dribbles_p90: 6.33,
tackles_p90: 1.84,
possession_won_p90: 1.70,
possession_lost_p90: 2.46,
fouls_committed_p90: 0.10,
offsides_p90: 0.00,
distance_covered_p90: 11.15,
distance_sprinted_p90: 3.20,
xt_bonus_p90: 0.52,
goals_p90_z: -0.34,
assists_p90_z: -0.50,
shots_p90_z: -0.50,
shot_accuracy_z: 0.00,
passes_p90_z: -0.50,
pass_accuracy_z: 0.32,
dribbles_p90_z: -0.50,
dribble_success_rate_z: 0.00,
tackles_p90_z: -0.50,
tackle_success_rate_z: 0.00,
offsides_p90_z: 0.52,
fouls_com

## Attacking Midfielders

In [391]:
# Creating the synthetic data

# Create an empty DataFrame with the specified columns
columns=["player", "goals", "assists", "shots", "shot_accuracy", "passes", "pass_accuracy", "dribbles", "dribble_success_rate", "tackles", "tackle_success_rate", "offsides", "fouls_committed", "possession_won", "possession_lost", "minutes_played", "distance_covered", "distance_sprinted", "half_length", "team_xg", "opponent_xg", "opponent_goals"]
pos_df=pd.DataFrame(columns=columns)

# Adding example wingback data manually for 4 performances
cam_test_data = [
    {
        "player": "The Masterclass (Prime KDB)",
        "minutes_played": 90, "half_length": 10,
        "goals": 1, "assists": 2, "shots": 3, "shot_accuracy": 66.0,
        "passes": 55, "pass_accuracy": 82.0, 
        "dribbles": 8, "dribble_success_rate": 75.0, 
        "tackles": 0, "tackle_success_rate": 0.0, 
        "possession_won": 1, "possession_lost": 4, 
        "distance_covered": 10.5, "distance_sprinted": 2.5,
        "fouls_committed": 0, "offsides": 0,
        "opponent_goals": 1, "team_xg": 3.0, "opponent_xg": 1.0
    },
    {
        "player": "The Disaster (The Turnover Machine)",
        "minutes_played": 60, "half_length": 10,
        "goals": 0, "assists": 0, "shots": 1, "shot_accuracy": 0.0,
        "passes": 18, "pass_accuracy": 60.0, 
        "dribbles": 5, "dribble_success_rate": 20.0, 
        "tackles": 0, "tackle_success_rate": 0.0, 
        "possession_won": 0, "possession_lost": 8, 
        "distance_covered": 6.0, "distance_sprinted": 1.1,
        "fouls_committed": 1, "offsides": 1,
        "opponent_goals": 2, "team_xg": 0.8, "opponent_xg": 2.2
    },
    {
        "player": "The Ghost (The Safe Passer)",
        "minutes_played": 90, "half_length": 10,
        "goals": 0, "assists": 0, "shots": 0, "shot_accuracy": 0.0,
        "passes": 35, "pass_accuracy": 92.0, 
        "dribbles": 1, "dribble_success_rate": 100.0, 
        "tackles": 1, "tackle_success_rate": 50.0, 
        "possession_won": 2, "possession_lost": 2, 
        "distance_covered": 10.0, "distance_sprinted": 1.5,
        "fouls_committed": 0, "offsides": 0,
        "opponent_goals": 1, "team_xg": 1.0, "opponent_xg": 1.0
    },
    {
        "player": "The Wrong Blueprint (The Destroyer at 10)",
        "minutes_played": 90, "half_length": 10,
        "goals": 0, "assists": 0, "shots": 0, "shot_accuracy": 0.0,
        "passes": 20, "pass_accuracy": 85.0, 
        "dribbles": 2, "dribble_success_rate": 50.0, 
        "tackles": 6, "tackle_success_rate": 80.0, 
        "possession_won": 7, "possession_lost": 1, 
        "distance_covered": 11.5, "distance_sprinted": 2.0,
        "fouls_committed": 2, "offsides": 0,
        "opponent_goals": 0, "team_xg": 0.5, "opponent_xg": 0.5
    }
]
# Add the test data to the empty dataframe (column names match the keys in the dictionaries)
for i in range(len(cam_test_data)):
    pos_df.loc[i] = cam_test_data[i]

print(pos_df)

                                      player  goals  assists  shots  \
0                The Masterclass (Prime KDB)      1        2      3   
1        The Disaster (The Turnover Machine)      0        0      1   
2                The Ghost (The Safe Passer)      0        0      0   
3  The Wrong Blueprint (The Destroyer at 10)      0        0      0   

   shot_accuracy  passes  pass_accuracy  dribbles  dribble_success_rate  \
0           66.0      55           82.0         8                  75.0   
1            0.0      18           60.0         5                  20.0   
2            0.0      35           92.0         1                 100.0   
3            0.0      20           85.0         2                  50.0   

   tackles  ...  fouls_committed  possession_won  possession_lost  \
0        0  ...                0               1                4   
1        0  ...                1               0                8   
2        1  ...                0               2             

In [392]:
# 2. Force them all to numeric in one swoop
cols_to_convert = pos_df.columns
# exempt the 'player' column since it's text
cols_to_convert = cols_to_convert.drop('player') 
pos_df[cols_to_convert] = pos_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
pos_df[cols_to_convert] = pos_df[cols_to_convert].fillna(0)

In [393]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = pos_df[pos_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    pos_df[perc_col] = pos_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

In [394]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    pos_df[col] = pos_df[col] * (h_base / pos_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    pos_df[f"{col}_p90"] = (pos_df[col] / (pos_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = means_stds_dict[f"{col}_p90"]["mean"]
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    pos_df[f"{col}_p90"] = ((pos_df[col] + dummy_stat) / (pos_df["minutes_played"] + dummy_minutes)) * 90.0

In [395]:
pos_df["xt_bonus_p90"] = 0.25 * (pos_df["distance_sprinted_p90"] / pos_df["distance_covered_p90"]) * np.log(pos_df["pass_accuracy"] * pos_df["passes_p90"] + 1)

In [396]:
# Match supremacy scalar
gamma = 0.2
delta_xg = (pos_df['team_xg'] + 1) / (pos_df['opponent_xg'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

In [397]:
# Now we import weights from root/workshop/position_weights.json
# Which is in the format {"RWB": {"goals_p90": 0.3, ...}, "CM": {...}, "ST": {...}}
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xt_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["CAM"]
final_weights = np.array([weights_dict[col] for col in col_names])

In [398]:
# Now we import the means and stds from root/workshop/position_means_stds.json
# Which is in the format {"CB": {"goals_p90": {"mean": 0.1, "std": 0.2}, ...}, "CM": {...}, "ST": {...}}
position_means_stds_path = project_root / "workshop" / "position_means_stds.json"
with open(position_means_stds_path, "r") as f:
    position_means_stds = json.load(f)
means_stds_dict = position_means_stds["CAM"]

In [399]:
# Compute final z-scores
negative_stats = ["fouls_committed_p90", "possession_lost_p90", "offsides_p90"]
for col in col_names:
    mean = means_stds_dict[col]["mean"]
    std = means_stds_dict[col]["std"]
    if std == 0:
        pos_df[f"{col}_z"] = 0
    elif col in negative_stats:
        pos_df[f"{col}_z"] = (mean - pos_df[col]) / std
    else:
        pos_df[f"{col}_z"] = (pos_df[col] - mean) / std

In [400]:
# ---------------------------------------------------------
# Step: Volume Masking (The Z-Score Override)
# If a player doesn't attempt enough actions, we neutralize 
# their efficiency Z-score to 0.0 so it doesn't affect their rating.
# ---------------------------------------------------------
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

# Map the raw volume column to its corresponding Z-score column
volume_z_pairs = {
    "passes": "pass_accuracy_z",
    "shots": "shot_accuracy_z",
    "dribbles": "dribble_success_rate_z",
    "tackles": "tackle_success_rate_z"
}

for vol_col, z_col in volume_z_pairs.items():
    # np.where(condition, value_if_true, value_if_false)
    # If volume < threshold, force Z-score to 0.0. Otherwise, keep the original Z-score.
    pos_df[z_col] = np.where(pos_df[vol_col] < volume_masks[vol_col], 0.0, pos_df[z_col])

In [401]:
# ---------------------------------------------------------
# 1. TACTICAL FLOORS (The Luxury Exemption)
# A CAM is explicitly told NOT to waste energy defending. 
# ---------------------------------------------------------
# We floor defensive stats at 0.0 so they take literally zero penalty for 0 tackles.
defensive_exemptions = ['tackles_p90_z', 'tackle_success_rate_z', 'possession_won_p90_z']
for col in defensive_exemptions:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.50, a_max=None)

# ---------------------------------------------------------
# 2. STANDARD CAPS (The Safety Nets)
# ---------------------------------------------------------
# Efficiency Volatility
efficiency_stats = ['dribble_success_rate_z', 'pass_accuracy_z', 'shot_accuracy_z']
for col in efficiency_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-0.75, a_max=None)

# Active Detriments (Fouls, Turnovers, Offsides)
detriment_stats = ['fouls_committed_p90_z', 'possession_lost_p90_z', 'offsides_p90_z']
for col in detriment_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-1.50, a_max=None)

# Passenger Cap (Volume Floors)
passenger_stats = [
    'passes_p90_z', 'dribbles_p90_z', 'shots_p90_z', 
    'distance_covered_p90_z', 'distance_sprinted_p90_z', 'xt_bonus_p90_z'
]
for col in passenger_stats:
    pos_df[col] = np.clip(pos_df[col], a_min=-1.0, a_max=None)

In [402]:
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = pos_df[z_score_cols].values
pos_df['raw_score'] = np.dot(z_matrix, final_weights)

In [403]:
# ---------------------------------------------------------
# 3. CONTEXTUAL BONUSES (The Maestro Identity)
# ---------------------------------------------------------
# Calculate Raw Score first
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = pos_df[z_score_cols].values
pos_df['raw_score'] = np.dot(z_matrix, final_weights)

# End Product Boost (CAMs are heavily judged on this)
pos_df['raw_score'] += pos_df['goals'] * 0.7   # Slight bump from standard 0.6
pos_df['raw_score'] += pos_df['assists'] * 0.9 # Slight bump from standard 0.8

# THE "MAESTRO" BONUS
# They orchestrated the game and split the defense wide open.
maestro_mask = (pos_df['xt_bonus_p90_z'] > 1.5) & (pos_df['passes_p90_z'] > 1.0)
pos_df['raw_score'] += np.where(maestro_mask, 0.40, 0)

# ATTACKING INVOLVEMENT
# Reward them for constantly shooting and driving the offense
pos_df['raw_score'] += np.where(pos_df['shots_p90_z'] > 1.5, (pos_df['shots_p90_z'] - 1.5) * 0.10, 0)

# ---------------------------------------------------------
# THE "METRONOME" BONUS (Fixes The Ghost)
# If a CAM plays it extremely safe (high accuracy, rarely loses the ball), 
# we give them a small retention bump so they aren't punished for lack of threat.
# Remember: For negative stats, a positive Z-score means they did it LESS (which is good!)
# ---------------------------------------------------------
metronome_mask = (pos_df['pass_accuracy_z'] > 1.0) & (pos_df['possession_lost_p90_z'] > 1.0)
pos_df['raw_score'] += np.where(metronome_mask, 0.15, 0)


# ---------------------------------------------------------
# THE "FALSE 6" BONUS (Fixes The Wrong Blueprint)
# If a CAM abandons the #10 role but puts on a defensive masterclass, 
# we manually inject points because the base weights ignore defending.
# ---------------------------------------------------------
false_6_mask = (pos_df['tackles_p90_z'] > 1.0) & (pos_df['possession_won_p90_z'] > 1.0)
pos_df['raw_score'] += np.where(false_6_mask, 0.60, 0)

In [404]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
pos_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (pos_df['raw_score'] - s_0))))

In [405]:
pos_df['final_rating'] = pos_df['raw_rating'] - match_supremacy_scalar

In [406]:
pos_df['final_rating'] = pos_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [428]:
random_row = pos_df.sample(n=1).iloc[0]
for col in pos_df.columns:
    if isinstance(random_row[col], float):
        print(f"{col}: {random_row[col]:.2f},")
    else:
        print(f"{col}: {random_row[col]},")

player: The Wrong Blueprint (The Destroyer at 10),
goals: 0.00,
assists: 0.00,
shots: 0.00,
shot_accuracy: 66.00,
passes: 20.00,
pass_accuracy: 85.00,
dribbles: 2.00,
dribble_success_rate: 47.50,
tackles: 6.00,
tackle_success_rate: 80.00,
offsides: 0.00,
fouls_committed: 2.00,
possession_won: 7.00,
possession_lost: 1.00,
minutes_played: 90,
distance_covered: 11.50,
distance_sprinted: 2.00,
half_length: 10,
team_xg: 0.50,
opponent_xg: 0.50,
opponent_goals: 0,
goals_p90: 0.00,
assists_p90: 0.00,
shots_p90: 0.00,
passes_p90: 21.28,
dribbles_p90: 4.94,
tackles_p90: 6.07,
possession_won_p90: 5.83,
possession_lost_p90: 1.67,
fouls_committed_p90: 1.53,
offsides_p90: 0.06,
distance_covered_p90: 11.74,
distance_sprinted_p90: 2.60,
xt_bonus_p90: 0.42,
goals_p90_z: -0.53,
assists_p90_z: -0.73,
shots_p90_z: -1.00,
shot_accuracy_z: 0.00,
passes_p90_z: -1.00,
pass_accuracy_z: 0.12,
dribbles_p90_z: -1.00,
dribble_success_rate_z: 0.00,
tackles_p90_z: 5.33,
tackle_success_rate_z: 2.15,
offsides_p90_z: 